In [ ]:
import pandas as pd
import numpy as np

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Utilities ---
import os
import warnings

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── File paths ────────────────────────────────────────────────────────────────
PARQUET_CLEAN    = "data/paysim_clean.parquet"      # INPUT  (Stage 3-4 output)
PARQUET_FEATURES = "data/paysim_features.parquet"   # OUTPUT (Stage 5)
PARQUET_TRAIN    = "data/train.parquet"             # OUTPUT (Stage 6)
PARQUET_TEST     = "data/test.parquet"              # OUTPUT (Stage 6)

# ── Display settings ─────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 60)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FRAUD_PALETTE = {0: "#2196F3", 1: "#F44336"}   # Blue = legit, Red = fraud

print("Imports and configuration complete.")

In [ ]:
assert os.path.exists(PARQUET_CLEAN), (
    f"\n❌ '{PARQUET_CLEAN}' not found.\n"
    "Run all cells in fraud_detection_stage3_4.ipynb first.\n"
    "The final cell in that notebook saves the cleaned dataset to this path."
)

df = pd.read_parquet(PARQUET_CLEAN)

print(f"✅ Loaded: {PARQUET_CLEAN}")
print(f"   Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns)}")
print(f"   Fraud  : {df['isFraud'].sum():,} ({df['isFraud'].mean()*100:.4f}%)")

df.head(3)

## Feature Engineering

In [ ]:
print("📐  Accounting Identity — Conceptual Anchor")
print()
print("  ORIGIN:  newbalanceOrig + amount − oldbalanceOrg  = 0  (should be)")
print("  DEST  :  oldbalanceDest + amount − newbalanceDest = 0  (should be)")
print()
print("  Non-zero result = the books don't balance = suspicious.")
print()
print("  PaySim naming quirk:")
print("    oldbalanceOrg  ← sender balance BEFORE  (ends in 'g')")
print("    newbalanceOrig ← sender balance AFTER   (ends in 'ig')")
print()
print("  We'll add 5 new columns to df in Cells 4–8 (no columns dropped).")

In [ ]:
df["errorBalanceOrig"] = (
    df["newbalanceOrig"].astype("float64")
    + df["amount"].astype("float64")
    - df["oldbalanceOrg"].astype("float64")
)

# Sanity check: distribution should differ markedly between fraud and legit
print("errorBalanceOrig — stats by fraud label:\n")
print(
    df.groupby("isFraud")["errorBalanceOrig"]
    .describe()
    .rename(index={0: "Legitimate (0)", 1: "Fraud (1)"})
    .round(2)
    .to_string()
)

print()
# Quick signal check: what fraction of each class has a large error?
threshold = 1.0   # anything > 1 local currency unit is a real discrepancy (not rounding)
fraud_large  = (df[df["isFraud"] == 1]["errorBalanceOrig"].abs() > threshold).mean()
legit_large  = (df[df["isFraud"] == 0]["errorBalanceOrig"].abs() > threshold).mean()
print(f"  |errorBalanceOrig| > {threshold}  in fraud rows : {fraud_large*100:.1f}%")
print(f"  |errorBalanceOrig| > {threshold}  in legit rows : {legit_large*100:.1f}%")
print()
print(" errorBalanceOrig added.")

In [ ]:
df["errorBalanceDest"] = (
    df["oldbalanceDest"].astype("float64")
    + df["amount"].astype("float64")
    - df["newbalanceDest"].astype("float64")
)

print("errorBalanceDest — stats by fraud label:\n")
print(
    df.groupby("isFraud")["errorBalanceDest"]
    .describe()
    .rename(index={0: "Legitimate (0)", 1: "Fraud (1)"})
    .round(2)
    .to_string()
)

print()
fraud_dest = (df[df["isFraud"] == 1]["errorBalanceDest"].abs() > threshold).mean()
legit_dest = (df[df["isFraud"] == 0]["errorBalanceDest"].abs() > threshold).mean()
print(f"  |errorBalanceDest| > {threshold}  in fraud rows : {fraud_dest*100:.1f}%")
print(f"  |errorBalanceDest| > {threshold}  in legit rows : {legit_dest*100:.1f}%")
print()
print(" errorBalanceDest added.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for col_idx, (col, title) in enumerate(
    [("errorBalanceOrig", "errorBalanceOrig"), ("errorBalanceDest", "errorBalanceDest")]
):
    data_legit = df[df["isFraud"] == 0][col]
    data_fraud = df[df["isFraud"] == 1][col]

    # Per-group clipping so extreme values in one group don't distort the other
    clip_lo = df[col].quantile(0.01)
    clip_hi = df[col].quantile(0.99)

    # Top row: histogram overlay
    ax_hist = axes[0][col_idx]
    ax_hist.hist(
        data_legit.clip(clip_lo, clip_hi), bins=80, alpha=0.55,
        color=FRAUD_PALETTE[0], density=True, label="Legitimate"
    )
    ax_hist.hist(
        data_fraud.clip(clip_lo, clip_hi), bins=80, alpha=0.65,
        color=FRAUD_PALETTE[1], density=True, label="Fraud"
    )
    ax_hist.set_title(f"{title} — Histogram", fontweight="bold")
    ax_hist.set_xlabel(col)
    ax_hist.set_ylabel("Density")
    ax_hist.legend()

    # Bottom row: boxplot
    ax_box = axes[1][col_idx]
    plot_data = pd.DataFrame({
        "value": pd.concat([
            data_legit.clip(clip_lo, clip_hi),
            data_fraud.clip(clip_lo, clip_hi)
        ]),
        "label": (
            ["Legitimate"] * len(data_legit) + ["Fraud"] * len(data_fraud)
        )
    })
    sns.boxplot(
        data=plot_data, x="label", y="value",
        palette={"Legitimate": FRAUD_PALETTE[0], "Fraud": FRAUD_PALETTE[1]},
        ax=ax_box, width=0.4
    )
    ax_box.set_title(f"{title} — Boxplot (clipped 1–99 pct)", fontweight="bold")
    ax_box.set_xlabel("")
    ax_box.set_ylabel(col)

plt.suptitle(
    "Error Features: Legitimate vs Fraud\n(values clipped at 1st–99th percentile for display)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

print("What to look for:")
print("  Wide separation between blue and red → strong discriminating feature.")
print("  Fraud median far from zero → accounting inconsistency is common in fraud.")

In [ ]:
df["flag_orig_zero_after"]  = (df["newbalanceOrig"] == 0).astype("int8")
df["flag_dest_zero_before"] = (df["oldbalanceDest"] == 0).astype("int8")
df["flag_dest_zero_both"]   = (
    (df["oldbalanceDest"] == 0) & (df["newbalanceDest"] == 0)
).astype("int8")

FLAG_COLS = ["flag_orig_zero_after", "flag_dest_zero_before", "flag_dest_zero_both"]

print("Zero-balance flag rates (%) by fraud label:\n")
flag_rates = (
    df.groupby("isFraud")[FLAG_COLS]
    .mean()
    .mul(100)
    .round(2)
    .rename(index={0: "Legitimate", 1: "Fraud"})
)
print(flag_rates.to_string())

# Visual: side-by-side bars showing flag fire rates per class
fig, ax = plt.subplots(figsize=(9, 4))
flag_rates.T.plot(kind="bar", color=[FRAUD_PALETTE[0], FRAUD_PALETTE[1]], ax=ax, width=0.6)
ax.set_title("Zero-Balance Flag Fire Rates by Class (%)", fontweight="bold")
ax.set_xlabel("Flag Feature")
ax.set_ylabel("Rate (%)")
ax.tick_params(axis="x", rotation=20)
ax.legend(["Legitimate", "Fraud"])
plt.tight_layout()
plt.show()

print()
print(" Zero-balance flag columns added.")

In [ ]:
type_dummies = pd.get_dummies(df["type"], prefix="type", drop_first=False, dtype="int8")
type_dummies = type_dummies.drop(columns=["type_CASH_OUT"])   # keep type_TRANSFER only

df = pd.concat([df, type_dummies], axis=1)

# We leave the original 'type' column in df for reference / debugging.
# FEATURE_COLS in Cell 9 will NOT include 'type' — only 'type_TRANSFER'.

print("type_TRANSFER column added:")
print(f"  = 1  (TRANSFER) : {df['type_TRANSFER'].sum():>9,} rows")
print(f"  = 0  (CASH_OUT) : {(df['type_TRANSFER'] == 0).sum():>9,} rows")
print()
print(f"  Fraud rate | TRANSFER : {df[df['type_TRANSFER']==1]['isFraud'].mean()*100:.4f}%")
print(f"  Fraud rate | CASH_OUT : {df[df['type_TRANSFER']==0]['isFraud'].mean()*100:.4f}%")
print()
print(" type_TRANSFER column added.")

In [ ]:
FEATURE_COLS = [
    "amount",
    "type_TRANSFER",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "errorBalanceOrig",
    "errorBalanceDest",
    "flag_orig_zero_after",
    "flag_dest_zero_before",
    "flag_dest_zero_both",
]

TARGET_COL = "isFraud"

X = df[FEATURE_COLS]
y = df[TARGET_COL]

print("Feature matrix defined:")
print(f"  X shape : {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"  y shape : {y.shape[0]:,} labels")
print(f"  Fraud rate in y : {y.mean()*100:.4f}%  ({y.sum():,} fraud / {len(y):,} total)")
print()
print("Features (in model order):")
for i, col in enumerate(FEATURE_COLS, 1):
    origin = "raw" if col in ["amount", "type_TRANSFER",
                               "oldbalanceOrg", "newbalanceOrig",
                               "oldbalanceDest", "newbalanceDest"] else "engineered"
    print(f"  {i:2d}.  {col:<26}  [{origin}]")
print(f"\n  Target: {TARGET_COL}")

In [ ]:
# ── Correlation bar chart ────────────────────────────────────────────────────
corr_with_target = (
    df[FEATURE_COLS + [TARGET_COL]]
    .corr()[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=abs, ascending=False)
)

print("Pearson correlation with isFraud (sorted by absolute value):\n")
for feat, val in corr_with_target.items():
    bar   = "█" * int(abs(val) * 50)
    sign  = "+" if val >= 0 else "−"
    stars = "  ★★★" if abs(val) > 0.15 else ("  ★★" if abs(val) > 0.08 else "")
    print(f"  {feat:<28} {sign}{abs(val):.4f}  {bar}{stars}")

print()
print("  Note: Low Pearson corr ≠ weak feature for tree models (non-linear signal).")

# ── Heatmap ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
corr_matrix = df[FEATURE_COLS + [TARGET_COL]].corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.4,
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title(
    "Feature Correlation Matrix (incl. isFraud)",
    fontweight="bold", fontsize=12
)
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("data", exist_ok=True)

COLS_TO_SAVE = ["step", "type"] + FEATURE_COLS + [TARGET_COL]
df[COLS_TO_SAVE].to_parquet(PARQUET_FEATURES, index=False)

size_mb = os.path.getsize(PARQUET_FEATURES) / 1024**2

print("=" * 62)
print("  STAGE 5 COMPLETE — FEATURE ENGINEERING SUMMARY")
print("=" * 62)
print(f"  Input rows           : {len(df):,}")
print(f"  Feature columns      : {len(FEATURE_COLS)}")
print()
print("  New columns added this stage:")
print("   errorBalanceOrig      — accounting error, sender side")
print("   errorBalanceDest      — accounting error, receiver side")
print("   flag_orig_zero_after  — sender account fully drained")
print("   flag_dest_zero_before — receiver account was empty before")
print("   flag_dest_zero_both   — receiver zero before AND after")
print("   type_TRANSFER         — transaction type as binary")
print()
print(f"  Saved → {PARQUET_FEATURES}  ({size_mb:.1f} MB)")
print("=" * 62)
print("\n  NEXT: Stage 6 — Train/Test Split (time-aware)")

## Splitting the Data

In [ ]:
print("Split strategy: TIME-AWARE (chronological by 'step')")
print()
print("  step 1 ──────────────── SPLIT_STEP ─────────────── step 744")
print("  |←──────────── TRAIN ──────────────→|←── TEST ─────→|")
print()
print("  Train: all rows where step ≤ SPLIT_STEP")
print("  Test : all rows where step > SPLIT_STEP")
print()
print("  Fraud detection predicts the future.")
print("  Time-aware split honestly simulates deployment.")

In [ ]:
TRAIN_FRAC  = 0.80
SPLIT_STEP  = int(df["step"].quantile(TRAIN_FRAC))

print(f"  PaySim step range   : {df['step'].min()} → {df['step'].max()}")
print(f"  Target train share  : {TRAIN_FRAC*100:.0f}% of transactions")
print(f"  Split step chosen   : step ≤ {SPLIT_STEP}  (80th pct by transaction volume)")
print(f"  Simulated equivalent: train ≈ days 1–{SPLIT_STEP//24}, "
      f"test ≈ days {SPLIT_STEP//24+1}–{df['step'].max()//24}")
print()

# Preview row counts
n_train_est = (df["step"] <= SPLIT_STEP).sum()
n_test_est  = (df["step"] >  SPLIT_STEP).sum()
print(f"  Estimated train rows : {n_train_est:>9,}  ({n_train_est/len(df)*100:.1f}%)")
print(f"  Estimated test rows  : {n_test_est:>9,}  ({n_test_est/len(df)*100:.1f}%)")

In [ ]:
train_df = df[df["step"] <= SPLIT_STEP].copy()
test_df  = df[df["step"] >  SPLIT_STEP].copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]

X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]

print(" Time-aware split complete.\n")
print(f"  X_train : {X_train.shape[0]:>9,} rows × {X_train.shape[1]} features")
print(f"  X_test  : {X_test.shape[0]:>9,} rows × {X_test.shape[1]} features")
print(f"  y_train : {y_train.shape[0]:>9,} labels")
print(f"  y_test  : {y_test.shape[0]:>9,} labels")

In [ ]:
total_rows = len(df)
split_rows = len(X_train) + len(X_test)
row_check  = "MATCH" if split_rows == total_rows else " MISMATCH"

train_max_step = train_df["step"].max()
test_min_step  = test_df["step"].min()
step_check     = " clean" if test_min_step > train_max_step else " TEMPORAL OVERLAP"

print("Split Validation:")
print(f"  Total rows (original)    : {total_rows:>9,}")
print(f"  Train + Test rows        : {split_rows:>9,}   {row_check}")
print()
print(f"  Train fraud rate         : {y_train.mean()*100:.4f}%  ({y_train.sum():,} frauds)")
print(f"  Test  fraud rate         : {y_test.mean()*100:.4f}%  ({y_test.sum():,} frauds)")
print()
print(f"  Max step in train        : {train_max_step}")
print(f"  Min step in test         : {test_min_step}")
print(f"  Temporal boundary        : {step_check}")

# ── Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: transaction volume over time, split boundary marked
vol_by_step = df.groupby("step").size()
axes[0].fill_between(vol_by_step.index, vol_by_step.values,
                     alpha=0.35, color="#607D8B", label="All transactions")
axes[0].axvline(SPLIT_STEP, color="black", linestyle="--", linewidth=1.5,
                label=f"Split at step {SPLIT_STEP}")
axes[0].fill_between(
    vol_by_step[vol_by_step.index <= SPLIT_STEP].index,
    vol_by_step[vol_by_step.index <= SPLIT_STEP].values,
    alpha=0.4, color=FRAUD_PALETTE[0], label="Train"
)
axes[0].fill_between(
    vol_by_step[vol_by_step.index > SPLIT_STEP].index,
    vol_by_step[vol_by_step.index > SPLIT_STEP].values,
    alpha=0.4, color=FRAUD_PALETTE[1], label="Test"
)
axes[0].set_title("Transaction Volume by Step", fontweight="bold")
axes[0].set_xlabel("Step (hour)")
axes[0].set_ylabel("Transactions per hour")
axes[0].legend(fontsize=8)

# Right: fraud count over time
fraud_by_step = df.groupby("step")["isFraud"].sum()
train_fraud   = fraud_by_step[fraud_by_step.index <= SPLIT_STEP]
test_fraud    = fraud_by_step[fraud_by_step.index >  SPLIT_STEP]

axes[1].bar(train_fraud.index, train_fraud.values,
            color=FRAUD_PALETTE[0], alpha=0.7, width=1, label="Train")
axes[1].bar(test_fraud.index,  test_fraud.values,
            color=FRAUD_PALETTE[1], alpha=0.7, width=1, label="Test")
axes[1].axvline(SPLIT_STEP, color="black", linestyle="--", linewidth=1.5)
axes[1].set_title("Fraud Events by Step", fontweight="bold")
axes[1].set_xlabel("Step (hour)")
axes[1].set_ylabel("Fraud count")
axes[1].legend()

plt.suptitle(f"Time-Aware Split at Step {SPLIT_STEP}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
n_neg  = int((y_train == 0).sum())
n_pos  = int((y_train == 1).sum())
SCALE_POS_WEIGHT = round(n_neg / n_pos, 2)

print("scale_pos_weight (for XGBoost — computed from y_train ONLY):\n")
print(f"  Legitimate in train : {n_neg:>9,}")
print(f"  Fraud in train      : {n_pos:>9,}")
print(f"  Ratio (neg / pos)   : {SCALE_POS_WEIGHT}")
print()
print(f"  → Stage 8 usage:  XGBClassifier(scale_pos_weight={SCALE_POS_WEIGHT})")
print()
print(f"  Interpretation: a missed fraud costs the model {SCALE_POS_WEIGHT:.0f}× more than")
print( "  a wrongly-blocked legitimate transaction during training.")

In [ ]:
print("📋 SMOTE status:")
print()
print("  X_train / y_train are UNMODIFIED — no resampling applied here.")
print()
print("  Resampling will be handled in Stage 8 via imblearn.pipeline.Pipeline,")
print("  which runs SMOTE inside each cross-validation fold to prevent leakage.")
print()
print("  Starting point for Stage 8:")
print(f"    Option A → XGBClassifier(scale_pos_weight={SCALE_POS_WEIGHT})   (try first)")
print( "    Option B → Pipeline([SMOTE(), XGBClassifier()])  (A/B against Option A)")
print()
print("  Primary evaluation metric: PR-AUC (Average Precision)")
print("  Secondary metrics: Recall, F1, Confusion Matrix")

In [ ]:
os.makedirs("data", exist_ok=True)

train_out          = X_train.copy()
train_out[TARGET_COL] = y_train.values

test_out           = X_test.copy()
test_out[TARGET_COL]  = y_test.values

train_out.to_parquet(PARQUET_TRAIN, index=False)
test_out.to_parquet(PARQUET_TEST,   index=False)

train_mb = os.path.getsize(PARQUET_TRAIN) / 1024**2
test_mb  = os.path.getsize(PARQUET_TEST)  / 1024**2

print(f" Saved train split → {PARQUET_TRAIN}  ({train_mb:.1f} MB)")
print(f" Saved test split  → {PARQUET_TEST}   ({test_mb:.1f} MB)")
print()
print("  How to reload in Stage 8:")
print("    train       = pd.read_parquet('data/train.parquet')")
print("    X_train     = train[FEATURE_COLS]")
print("    y_train     = train['isFraud']")
print()
print("    test        = pd.read_parquet('data/test.parquet')")
print("    X_test      = test[FEATURE_COLS]")
print("    y_test      = test['isFraud']")

In [ ]:
# =============================================================================
# %% CELL 19 — Stage 6 Checkpoint
# =============================================================================

print("=" * 62)
print("  STAGE 6 COMPLETE — TRAIN/TEST SPLIT SUMMARY")
print("=" * 62)
print(f"  Split strategy        : Time-aware (step ≤ {SPLIT_STEP} = train)")
print(f"  Train rows            : {len(X_train):>9,}  ({len(X_train)/len(df)*100:.1f}%)")
print(f"  Test  rows            : {len(X_test):>9,}  ({len(X_test)/len(df)*100:.1f}%)")
print(f"  Train fraud rate      : {y_train.mean()*100:.4f}%  ({y_train.sum():,} frauds)")
print(f"  Test  fraud rate      : {y_test.mean()*100:.4f}%  ({y_test.sum():,} frauds)")
print(f"  scale_pos_weight      : {SCALE_POS_WEIGHT}  (for XGBoost imbalance handling)")
print()
print("  Decisions made this stage:")
print("   Time-aware split      — no temporal leakage")
print("   step excluded from X  — simulation hour is not a real feature")
print("   scale_pos_weight from y_train only — no look-ahead")
print("   SMOTE deferred to Stage 8 pipeline — correct leakage discipline")
print()
print("  Files saved:")
print(f"   {PARQUET_FEATURES}")
print(f"   {PARQUET_TRAIN}")
print(f"   {PARQUET_TEST}")
print("=" * 62)
print()
print("  NEXT: Stage 7 — Class Imbalance + Stage 8 — Model Training")
print()
print("  Pipeline for Stage 8:")
print("  ┌──────────────────────────────────────────────────────┐")
print("  │  train = pd.read_parquet('data/train.parquet')       │")
print("  │  X_train = train[FEATURE_COLS]                       │")
print("  │  y_train = train['isFraud']                          │")
print("  │                                                      │")
print("  │  Pipeline([                                          │")
print("  │      ('smote', SMOTE()),           ← Stage 7         │")
print("  │      ('model', XGBClassifier(                        │")
print(f"  │          scale_pos_weight={SCALE_POS_WEIGHT}))          │")
print("  │  ]).fit(X_train, y_train)          ← Stage 8         │")
print("  │                                                      │")
print("  │  Evaluate on X_test, y_test with PR-AUC, Recall, F1  │")
print("  └──────────────────────────────────────────────────────┘")
